# Figure 1, panel 2
Open from this panel directory. The isolated ring is left and the sulfur-containing fragment is right. SVG and PNG exports have transparent backgrounds. Install librsvg for rsvg-convert.


In [ ]:
from pathlib import Path
import io,re,shutil,subprocess
import xml.etree.ElementTree as ET
from IPython.display import SVG,Image,display
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D
DATA = Path("../../data/cdk1_5lqf_candidate03")
OUTPUT = Path("output")
TEAL = "#007780"
ORANGE = "#b74725"
CANVAS_WIDTH,CANVAS_HEIGHT = 1200,720
BOND_LINE_WIDTH = 9
ATOM_FONT_SIZE = 86
PADDING = 0.075
PNG_WIDTH = 3600
ROTATE_DEGREES = 0
MIRROR_HORIZONTAL = True
SHOW_ATTACHMENT_MARKERS = False
ATTACHMENT_MARKER_RADIUS = 10
CUSTOM_ATOM_LABELS = {}
FRAGMENTS_ONLY = True
RETAINED_ATOMS = set(range(16))
ANCHOR_ATOMS = (0,6)
ET.register_namespace("", "http://www.w3.org/2000/svg")


In [ ]:
def load_one_sdf(path):
    molecules = list(Chem.SDMolSupplier(str(path), removeHs=False))
    if len(molecules) != 1 or molecules[0] is None:
        raise ValueError('Expected one readable molecule: ' + path.as_posix())
    return molecules[0]

def atom_indices(element):
    return [int(value) for value in re.findall('atom-(\\d+)', element.get('class', ''))]

def classify_atom_join(element, drawer, molecule):
    if atom_indices(element) or element.tag.rsplit('}', 1)[-1] != 'path':
        return
    style = element.get('style', '')
    if 'stroke:' not in style or 'stroke:none' in style:
        return
    numeric = '[-+]?(?:\\d*\\.\\d+|\\d+)(?:[eE][-+]?\\d+)?'
    data = element.get('d', '')
    join_pattern = f'\\s*M\\s*{numeric}[,\\s]+{numeric}\\s*L\\s*{numeric}[,\\s]+{numeric}\\s*L\\s*{numeric}[,\\s]+{numeric}\\s*'
    if re.fullmatch(join_pattern, data) is None:
        raise ValueError('Unclassified molecular stroke: inspect the RDKit SVG before using the drawing.')
    values = [float(value) for value in re.findall(numeric, data)]
    (center_x, center_y) = values[2:4]
    distances = []
    for index in range(molecule.GetNumAtoms()):
        point = drawer.GetDrawCoords(index)
        distances.append(((point.x - center_x) ** 2 + (point.y - center_y) ** 2, index))
    (distance_squared, index) = min(distances)
    if distance_squared > 0.1 ** 2:
        raise ValueError('A bond join cannot be assigned to an atom at the saved drawing coordinates.')
    element.set('class', 'atom-join atom-' + str(index))

def make_panel_svg(mol, fragments_only=False):
    mol = Chem.Mol(mol)
    if MIRROR_HORIZONTAL:
        conformer = mol.GetConformer()
        for index in range(mol.GetNumAtoms()):
            point = conformer.GetAtomPosition(index)
            conformer.SetAtomPosition(index, (-point.x, point.y, point.z))
    drawer = rdMolDraw2D.MolDraw2DSVG(CANVAS_WIDTH, CANVAS_HEIGHT)
    options = drawer.drawOptions()
    options.clearBackground = False
    options.padding = PADDING
    options.bondLineWidth = BOND_LINE_WIDTH
    options.fixedFontSize = ATOM_FONT_SIZE
    options.rotate = ROTATE_DEGREES
    options.useBWAtomPalette()
    for (index, label) in CUSTOM_ATOM_LABELS.items():
        options.atomLabels[int(index)] = str(label)
    drawer.DrawMolecule(mol)
    drawer.FinishDrawing()
    tree = ET.fromstring(drawer.GetDrawingText())
    for element in list(tree):
        classify_atom_join(element, drawer, mol)
        indices = atom_indices(element)
        if not indices:
            continue
        if fragments_only and (not all((i in RETAINED_ATOMS for i in indices))):
            tree.remove(element)
            continue
        color = TEAL if all((i in RETAINED_ATOMS for i in indices)) else ORANGE
        style = element.get('style', '')
        style = re.sub('(?<=stroke:)#[A-Fa-f0-9]{6}', color, style)
        style = re.sub('(?<=fill:)#[A-Fa-f0-9]{6}', color, style)
        element.set('style', style)
        if element.get('fill') and element.get('fill') != 'none':
            element.set('fill', color)
    if SHOW_ATTACHMENT_MARKERS:
        for index in ANCHOR_ATOMS:
            point = drawer.GetDrawCoords(index)
            ET.SubElement(tree, '{http://www.w3.org/2000/svg}circle', {'cx': str(point.x), 'cy': str(point.y), 'r': str(ATTACHMENT_MARKER_RADIUS), 'fill': 'none', 'stroke': TEAL, 'stroke-width': str(BOND_LINE_WIDTH / 2), 'class': 'attachment-marker atom-' + str(index)})
    return ET.tostring(tree, encoding='unicode')

def transparent_png(svg_text):
    converter = shutil.which('rsvg-convert')
    if converter is None:
        raise RuntimeError('Install librsvg / rsvg-convert to display the PNG; the SVG needs no converter.')
    return subprocess.run([converter, '--format', 'png', '--width', str(PNG_WIDTH)], input=svg_text.encode('utf-8'), stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True).stdout


In [ ]:
molecule = load_one_sdf(DATA / "ligand_2d_depiction.sdf")
SVG_TEXT = make_panel_svg(molecule, fragments_only=FRAGMENTS_ONLY)
display(SVG(SVG_TEXT))
PNG_BYTES = transparent_png(SVG_TEXT)
display(Image(data=PNG_BYTES, format="png", width=900))


In [ ]:
OUTPUT.mkdir(exist_ok=True)
(OUTPUT / "panel2.svg").write_text(SVG_TEXT)
(OUTPUT / "panel2.png").write_bytes(PNG_BYTES)
